In [1]:
import os
import shutil
import subprocess
import string
import re
import random
import pandas as pd
import numpy as np

In [2]:
#Compiling Magrathea is done here
def compile_magrathea(magrathea_source_dir):
    """
    Run 'make' in the Magrathea source directory to compile the planet executable
    
    Parameters:
    - magrathea_source_dir: path to Magrathea source code
    
    Returns:
    - success: boolean indicating if compilation was successful
    """

    
    if not os.path.exists(magrathea_source_dir):
        print(f"Source directory does not exist: {magrathea_source_dir}")
        return False, "Directory not found"
    
    result = subprocess.run(['make'], 
                           cwd=magrathea_source_dir,
                           capture_output=True, 
                           text=True)
    
    # Check if compilation was successful
    planet_executable = os.path.join(magrathea_source_dir, "planet")
    
    if result.returncode == 0 and os.path.isfile(planet_executable):
        
        return True
    else:
        return False

In [3]:
def run_magrathea_simulation(magrathea_source_dir, run_dir, mode, timeout):
    """Run a Magrathea simulation from a prepared run directory

    run_dir = directory with your running configuration file


    """
    
    # Let us first check if the source dir exists
    planet_exe = os.path.join(magrathea_source_dir, "planet")
    if not os.path.isfile(planet_exe):
        print(f"Planet executable not found at {planet_exe}")
        return False, "", "Executable not found"
    
    # Get relative path to config file
    run_mode_file = os.path.join(run_dir, f"mode{mode}.cfg")
    rel_run_path = os.path.relpath(run_mode_file, magrathea_source_dir)
    
    print(f"\n Running: ./planet {rel_run_path}")
    print(f"   Working dir: {magrathea_source_dir}")
    
    # Run the simulation
    try:
        
        result = subprocess.run(['./planet',  rel_run_path],
                               cwd=magrathea_source_dir,
                               capture_output=True,
                               text=True,
                               timeout=timeout)
        
        # Save output
        with open(os.path.join(run_dir, "run_output.log"), 'w') as f:
            f.write(f"Return code: {result.returncode}\n\n")
            f.write("=== STDOUT ===\n")
            f.write(result.stdout)
            f.write("\n=== STDERR ===\n")
            f.write(result.stderr)
        
        if result.returncode == 0:
            return True
        else:
            print(f" Simulation failed")
            return False
            
    except subprocess.TimeoutExpired:
        print(f"Simulation timed out after {timeout} seconds")
        return False, "", "Timeout"
    except Exception as e:
        print(f"Error: {e}")
        return False, "", str(e)

In [4]:
def generate_random_dirname(length=16):
    """Generate a simple random directory name"""
    random_str = ''.join(random.choices(string.ascii_lowercase + string.digits, k=length))
    return f"{random_str}"

In [5]:
def create_mode4_input(filepath, mass, radius):
    """
    Write planet input file with mass and radius data
    
    Parameters:
    - filepath: where to save the file
    - mass: single value or list of masses (Earth masses)
    - radius: single value or list of radii (Earth radii)
    - comment: optional comment for the file
    """
    
    # Convert single values to lists for uniform processing
    if not isinstance(mass, (list, tuple)):
        mass = [mass]
    if not isinstance(radius, (list, tuple)):
        radius = [radius]
    
    # Check if lengths match
    if len(mass) != len(radius):
        raise ValueError(f"mass and radius must have same length! Got {len(mass)} masses and {len(radius)} radii")
    
    with open(filepath, 'w') as f:
        # Write header with comment
        f.write("M (Earth-masses)\tR (Earth-radii)\n")
        
        # Write all data points
        for m, r in zip(mass, radius):
            f.write(f"{m}\t{r}\n")
    
    # Print summary
    print(f"Created input file: {filepath}")
    print(f"   {len(mass)} data point(s) written")
    if len(mass) > 1:
        print(f"   Mass range: {min(mass)} - {max(mass)} Earth masses")
        print(f"   Radius range: {min(radius)} - {max(radius)} Earth radii")

In [6]:
def create_mode0_config(template_path, run_dir, **kwargs):
    """
    Create a modified Mode 4 config file
    
    Parameters:
    - template_path: path to original config file
    - output_dir: directory to save new config
    - **kwargs: parameters to change
    """
    
    os.makedirs(run_dir, exist_ok=True)
    
    # Read template
    with open(template_path, 'r') as f:
        config_content = f.read()
    
    # Define mapping between parameter names and their patterns in the file
    param_map = {
        # Mode 4 parameters
        'mass_of_core': r'mass_of_core=[\d.]+',
        'mass_of_mantle': r'mass_of_mantle=[\d.]+',
        'mass_of_hydro': r'mass_of_hydro=[\d.]+',
        'mass_of_atm': r'mass_of_atm=[\d.]+',
        'surface_temp': r'surface_temp=[\d.]+',
        
        # Temperature discontinuity jumps
        'temp_jump_1': r'temp_jump_1=[\d.]+',  # Atmosphere to Hydrosphere
        'temp_jump_2': r'temp_jump_2=[\d.]+',  # Hydrosphere to Mantle
        'temp_jump_3': r'temp_jump_3=[\d.]+',  # Mantle to Core
        
        # Global Run Options
        'P_surface': r'P_surface=[\d.Ee+-]+',   # Surface pressure in Pa
        
        # File paths
        'output_file': r'output_file="[^"]*"'
    }
    
    # Replace each parameter if provided
    for param, value in kwargs.items():
        if param in param_map:
            pattern = param_map[param]
            if isinstance(value, str):
                if param in ['output_file']:
                    replacement = f'{param}="{value}"'
                else:
                    replacement = f'{param}={value}'
            else:
                replacement = f'{param}={value}'
            
            config_content = re.sub(pattern, replacement, config_content)
        else:
            print(f"Warning: Unknown parameter '{param}'")
    
    # Save new config
    output_path = os.path.join(run_dir, "mode0.cfg")
    with open(output_path, 'w') as f:
        f.write(config_content)
    
    print(f"\nConfig saved to: {output_path}")
    return output_path

In [7]:
def create_mode4_config(template_path, run_dir, **kwargs):
    """
    Create a modified Mode 4 config file
    
    Parameters:
    - template_path: path to original config file
    - output_dir: directory to save new config
    - **kwargs: parameters to change
    """
    
    os.makedirs(run_dir, exist_ok=True)
    
    # Read template
    with open(template_path, 'r') as f:
        config_content = f.read()
    
    # Define mapping between parameter names and their patterns in the file
    param_map = {
        # Mode 4 parameters
        'find_layer': r'find_layer=\d+',
        'layer_outer': r'layer_outer=\d+',
        'layer_inner': r'layer_inner=\d+',
        'PMR_min': r'PMR_min=[\d.]+',
        'PMR_max': r'PMR_max=[\d.]+',
        'PMR_step': r'PMR_step=[\d.]+',
        'R_error': r'R_error=[\d.]+',
        'surface_temp': r'surface_temp=[\d.]+',
        
        # Temperature discontinuity jumps
        'temp_jump_1': r'temp_jump_1=[\d.]+',  # Atmosphere to Hydrosphere
        'temp_jump_2': r'temp_jump_2=[\d.]+',  # Hydrosphere to Mantle
        'temp_jump_3': r'temp_jump_3=[\d.]+',  # Mantle to Core
        
        # Global Run Options
        'P_surface': r'P_surface=[\d.Ee+-]+',   # Surface pressure in Pa
        
        # File paths
        'input_file': r'input_file="[^"]*"',
        'output_file': r'output_file="[^"]*"'
    }
    
    # Replace each parameter if provided
    for param, value in kwargs.items():
        if param in param_map:
            pattern = param_map[param]
            if isinstance(value, str):
                if param in ['input_file', 'output_file']:
                    replacement = f'{param}="{value}"'
                else:
                    replacement = f'{param}={value}'
            else:
                replacement = f'{param}={value}'
            
            config_content = re.sub(pattern, replacement, config_content)
        else:
            print(f"Warning: Unknown parameter '{param}'")
    
    # Save new config
    output_path = os.path.join(run_dir, "mode4.cfg")
    with open(output_path, 'w') as f:
        f.write(config_content)
    
    print(f"\nConfig saved to: {output_path}")
    return output_path

In [8]:
def setting_up_dir(run_dir, input_dir, result_dir):
    """Sets up the directories, by adding random directory names, wich will be used later"""

    
    random_name = generate_random_dirname()
    
    new_run_dir = os.path.join(run_dir, random_name)
    new_input_dir = os.path.join(input_dir, random_name)
    new_result_dir = os.path.join(result_dir, random_name)

    os.makedirs(new_run_dir, exist_ok=True)
    os.makedirs(new_input_dir, exist_ok=True)
    os.makedirs(new_result_dir, exist_ok=True)
    
    return new_run_dir, new_input_dir, new_result_dir

In [9]:
def save_magrathea_results(magrathea_source,mode,output_file,**kwargs):
    """
    Save Magrathea results with all parameters passed as kwargs
    
    Required kwargs:
    - output_file: path to the Magrathea output file
    - run_name: name for this run
    
    All other kwargs will be saved as parameters
    """
    
    # Extract required parameters
    
    if output_file is None:
        raise ValueError("Must provide output_file")
    
    # Create save directory
    save_dir = os.path.join(magrathea_source, f"result_final{mode}")
    os.makedirs(save_dir, exist_ok=True)
    final_file = os.path.join(save_dir, f"results.xlsx")
    file_exists = os.path.isfile(final_file)
    if file_exists:
        df_first_results = pd.read_excel(final_file , sheet_name='Results')
        df_first_parameters = pd.read_excel(final_file , sheet_name='Run_Parameters')
    # Read the output data
    try:
        # Try space-separated first
        df_results = pd.read_csv(output_file, sep='\s+', comment='#')
    except:
        try:
            # Try tab-separated
            df_results = pd.read_csv(output_file, sep='\t', comment='#')
        except:
            # Manual parsing as fallback
            data = []
            with open(output_file, 'r') as f:
                lines = f.readlines()
            
            # Find header
            for i, line in enumerate(lines):
                if line.strip() and not line.startswith('#'):
                    headers = line.strip().split()
                    for line2 in lines[i+1:]:
                        if line2.strip() and not line2.startswith('#'):
                            values = line2.strip().split()
                            if len(values) == len(headers):
                                data.append(dict(zip(headers, map(float, values))))
                    break
            df_results = pd.DataFrame(data)

    
    params_data = {
        **kwargs  
    }
    df_params = pd.DataFrame([params_data])
    df_params = pd.concat([df_params] * len(df_results), ignore_index=True)
    if file_exists:
        df_results = pd.concat([df_first_results, df_results], ignore_index=True)
        df_params = pd.concat([df_first_parameters, df_params], ignore_index=True)
    # Save to Excel with multiple sheets
    excel_path = os.path.join(save_dir, f"results.xlsx")
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Results sheet (many rows)
        df_results.to_excel(writer, sheet_name='Results', index=False)
        
        # Parameters sheet (ONE row per run)
        df_params.to_excel(writer, sheet_name='Run_Parameters', index=False)
        
        # Summary sheet
    
    return save_dir

<>:28: SyntaxWarning: invalid escape sequence '\s'
<>:28: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_837/1312533436.py:28: SyntaxWarning: invalid escape sequence '\s'
  df_results = pd.read_csv(output_file, sep='\s+', comment='#')


In [10]:
def save_magrathea_boundary_mode0(magrathea_source,mode,output_file,**kwargs):
    """
    Save Magrathea results with all parameters passed as kwargs
    
    Required kwargs:
    - output_file: path to the Magrathea output file
    - run_name: name for this run
    
    All other kwargs will be saved as parameters
    """
    
    # Extract required parameters
    
    if output_file is None:
        raise ValueError("Must provide output_file")
    
    # Create save directory
    save_dir = os.path.join(magrathea_source, f"result_final{mode}")
    os.makedirs(save_dir, exist_ok=True)
    final_file = os.path.join(save_dir, f"results.xlsx")
    file_exists = os.path.isfile(final_file)
    if file_exists:
        df_first_results = pd.read_excel(final_file , sheet_name='Results')
        df_first_parameters = pd.read_excel(final_file , sheet_name='Run_Parameters')
    # Read the output data
    try:
        # Try space-separated first
        df_results = pd.read_csv(output_file, sep='\s{2,}', comment='#')
    except:
        try:
            # Try tab-separated
            df_results = pd.read_csv(output_file, sep='\t', comment='#')
        except:
            # Manual parsing as fallback
            data = []
            with open(output_file, 'r') as f:
                lines = f.readlines()
            
            # Find header
            for i, line in enumerate(lines):
                if line.strip() and not line.startswith('#'):
                    headers = line.strip().split()
                    for line2 in lines[i+1:]:
                        if line2.strip() and not line2.startswith('#'):
                            values = line2.strip().split()
                            if len(values) == len(headers):
                                data.append(dict(zip(headers, map(float, values))))
                    break
            df_results = pd.DataFrame(data)

    mass_of_mantle = kwargs['mass_of_mantle']
    mass_of_core = kwargs['mass_of_core']
    mass_of_rock = mass_of_mantle + mass_of_core
    idx = (df_results["M (earth)"] - mass_of_rock).abs().idxmin()

    df_results = df_results.loc[[idx]]
    
    params_data = {
        **kwargs  
    }
    df_params = pd.DataFrame([params_data])
    if file_exists:
        df_results = pd.concat([df_first_results, df_results], ignore_index=True)
        df_params = pd.concat([df_first_parameters, df_params], ignore_index=True)

    # Save to Excel with multiple sheets
    excel_path = os.path.join(save_dir, f"results.xlsx")
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Results sheet (many rows)
        df_results.to_excel(writer, sheet_name='Results', index=False)
        
        # Parameters sheet (ONE row per run)
        df_params.to_excel(writer, sheet_name='Run_Parameters', index=False)
        
        # Summary sheet
    
    return save_dir

<>:28: SyntaxWarning: invalid escape sequence '\s'
<>:28: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_837/971988015.py:28: SyntaxWarning: invalid escape sequence '\s'
  df_results = pd.read_csv(output_file, sep='\s{2,}', comment='#')


In [11]:
def setup_run(magrathea_source, mode, **kwargs):
    """
    magrathea_source : magrathea source code
    """
    # Create the different directories, normally they exist already
    run_dir = os.path.join(magrathea_source, "run_test")
    input_dir = os.path.join(magrathea_source, "input_test")
    result_dir = os.path.join(magrathea_source, "result_test")

    os.makedirs(run_dir, exist_ok=True)
    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(result_dir, exist_ok=True)
    
    # Create input and output files
    new_run_dir, new_input_dir, new_result_dir = setting_up_dir(run_dir, input_dir, result_dir)
    input_file = os.path.join(new_input_dir, "input.txt")
    output_file = os.path.join(new_result_dir, "output.txt")
    if mode==4:
        mass = kwargs.pop('mass', [1.0])
        radius = kwargs.pop('radius', [1.0])
        create_mode4_input(input_file, mass, radius)
    
        # Update kwargs for config
        kwargs['input_file'] = input_file
        kwargs['output_file'] = output_file
    
        # Create config
        original_run_dir = os.path.join(magrathea_source, "run")
        original_config = os.path.join(original_run_dir, f"mode{mode}.cfg")
        create_mode4_config(original_config, new_run_dir, **kwargs)

        #Compile and run Magrathea
        compile_magrathea(magrathea_source)
        run_magrathea_simulation(magrathea_source, new_run_dir,mode, 5000)

        #Save the results, we will update an excel-file
        if kwargs.get('input_file'):
            del kwargs['input_file'] 
        if kwargs.get('output_file'):
            del kwargs['output_file'] 
        save_magrathea_results(magrathea_source,mode, output_file, **kwargs)
        
    elif mode==0:

        kwargs['output_file'] = output_file

        original_run_dir = os.path.join(magrathea_source, "run")
        original_config = os.path.join(original_run_dir, f"mode{mode}.cfg")
        create_mode0_config(original_config, new_run_dir, **kwargs)
                
        compile_magrathea(magrathea_source)
        run_magrathea_simulation(magrathea_source, new_run_dir, mode, 5000)

        if kwargs.get('output_file'):
            del kwargs['output_file'] 
        save_magrathea_boundary_mode0(magrathea_source, mode, output_file, **kwargs)
        #Finale remove the other directories
        shutil.rmtree(new_run_dir)
        shutil.rmtree(new_input_dir)
        shutil.rmtree(new_result_dir)
        
    return run_dir

In [37]:
Temperatures = np.linspace(50,150,101)
print(Temperatures)

[ 50.  51.  52.  53.  54.  55.  56.  57.  58.  59.  60.  61.  62.  63.
  64.  65.  66.  67.  68.  69.  70.  71.  72.  73.  74.  75.  76.  77.
  78.  79.  80.  81.  82.  83.  84.  85.  86.  87.  88.  89.  90.  91.
  92.  93.  94.  95.  96.  97.  98.  99. 100. 101. 102. 103. 104. 105.
 106. 107. 108. 109. 110. 111. 112. 113. 114. 115. 116. 117. 118. 119.
 120. 121. 122. 123. 124. 125. 126. 127. 128. 129. 130. 131. 132. 133.
 134. 135. 136. 137. 138. 139. 140. 141. 142. 143. 144. 145. 146. 147.
 148. 149. 150.]


In [38]:
for T in Temperatures:
    run_dir = setup_run(
        magrathea_source="/home/mathias/Margtest/Magrathea",
        mode=4,
    
        # Multiple mass/radius points
        mass=[13],
        radius=[3.2],
    
        # Config parameters
        find_layer=3,
        layer_outer=2,
        layer_inner=1,
        PMR_min=100,
        PMR_max=100,
        PMR_step=1.0,
        surface_temp=T,
        temp_jump_1=0,
        temp_jump_2=0,
        temp_jump_3=0,
        P_surface=1.0e5
    )

Created input file: /home/mathias/Margtest/Magrathea/input_test/vjjyyd0w4ldqr37n/input.txt
   1 data point(s) written

Config saved to: /home/mathias/Margtest/Magrathea/run_test/vjjyyd0w4ldqr37n/mode4.cfg

 Running: ./planet run_test/vjjyyd0w4ldqr37n/mode4.cfg
   Working dir: /home/mathias/Margtest/Magrathea


BadZipFile: File is not a zip file

In [36]:
filename = f"/home/mathias/Margtest/Magrathea/result_final4/results.xlsx"
mask = pd.read_excel(filename, sheet_name="Run_Parameters")["find_layer"] == 3

Core_Mass = pd.read_excel(filename, sheet_name="Results")["MCore"]
Core_Mass = Core_Mass[mask]

Mantle_Mass = pd.read_excel(filename, sheet_name="Results")["MMantle"]
Mantle_Mass = Mantle_Mass[mask]

Hydro_Mass = pd.read_excel(filename, sheet_name="Results")["MWater"]
Hydro_Mass = Hydro_Mass[mask]

Atm_Mass = pd.read_excel(filename, sheet_name="Results")["MGas"]
Atm_Mass = Atm_Mass[mask]

Temp_surface = pd.read_excel(filename, sheet_name="Run_Parameters")["surface_temp"]
Temp_surface = Temp_surface[mask]

Temp_jump_1 = pd.read_excel(filename, sheet_name="Run_Parameters")["temp_jump_1"] 
Temp_jump_2 = pd.read_excel(filename, sheet_name="Run_Parameters")["temp_jump_2"]
Temp_jump_3 = pd.read_excel(filename, sheet_name="Run_Parameters")["temp_jump_3"]
Temp_jump_1 = Temp_jump_1[mask]
Temp_jump_2 = Temp_jump_2[mask]
Temp_jump_3 = Temp_jump_3[mask]

Press_surface = pd.read_excel(filename, sheet_name="Run_Parameters")["P_surface"]
Press_surface = Press_surface[mask]

new_mask = Hydro_Mass < 5

Core_Mass = Core_Mass[new_mask].reset_index(drop=True)
Mantle_Mass = Mantle_Mass[new_mask].reset_index(drop=True)
Hydro_Mass = Hydro_Mass[new_mask].reset_index(drop=True)
Atm_Mass = Atm_Mass[new_mask].reset_index(drop=True)
Temp_surface = Temp_surface[new_mask].reset_index(drop=True)
Temp_jump_1 = Temp_jump_1[new_mask].reset_index(drop=True)
Temp_jump_2 = Temp_jump_2[new_mask].reset_index(drop=True)
Temp_jump_3 = Temp_jump_3[new_mask].reset_index(drop=True)
Press_surface = Press_surface[new_mask].reset_index(drop=True)

print(Hydro_Mass)

BadZipFile: File is not a zip file

In [40]:
for i in range(len(Temperatures)):
    run_dir = setup_run(
        magrathea_source="/home/mathias/Margtest/Magrathea",
        mode=0,
    
        # Multiple mass/radius point
        # Config parameters
        mass_of_core=0,
        mass_of_mantle=9,
        mass_of_hydro=3,
        mass_of_atm=1.5,
        surface_temp=Temperatures[i],
        temp_jump_1=0,
        temp_jump_2=2000,
        temp_jump_3=0,
        P_surface=100000
        )


Config saved to: /home/mathias/Margtest/Magrathea/run_test/nw82dcws713n49wb/mode0.cfg

 Running: ./planet run_test/nw82dcws713n49wb/mode0.cfg
   Working dir: /home/mathias/Margtest/Magrathea


/tmp/ipykernel_837/971988015.py:28: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_results = pd.read_csv(output_file, sep='\s{2,}', comment='#')



Config saved to: /home/mathias/Margtest/Magrathea/run_test/lzljivphyogpy2z7/mode0.cfg

 Running: ./planet run_test/lzljivphyogpy2z7/mode0.cfg
   Working dir: /home/mathias/Margtest/Magrathea


/tmp/ipykernel_837/971988015.py:28: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_results = pd.read_csv(output_file, sep='\s{2,}', comment='#')



Config saved to: /home/mathias/Margtest/Magrathea/run_test/f5b9yv4vd1aa96cu/mode0.cfg

 Running: ./planet run_test/f5b9yv4vd1aa96cu/mode0.cfg
   Working dir: /home/mathias/Margtest/Magrathea


/tmp/ipykernel_837/971988015.py:28: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_results = pd.read_csv(output_file, sep='\s{2,}', comment='#')


... (192 outputs left out)


Config saved to: /home/mathias/Margtest/Magrathea/run_test/fkkukszcla8fd8dm/mode0.cfg

 Running: ./planet run_test/fkkukszcla8fd8dm/mode0.cfg
   Working dir: /home/mathias/Margtest/Magrathea


/tmp/ipykernel_837/971988015.py:28: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_results = pd.read_csv(output_file, sep='\s{2,}', comment='#')



Config saved to: /home/mathias/Margtest/Magrathea/run_test/yu429f1xgjxwyze8/mode0.cfg

 Running: ./planet run_test/yu429f1xgjxwyze8/mode0.cfg
   Working dir: /home/mathias/Margtest/Magrathea


/tmp/ipykernel_837/971988015.py:28: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_results = pd.read_csv(output_file, sep='\s{2,}', comment='#')
